# Homework 6: Supervised Learning

This notebook solves the campus store purchase classification task using five supervised learning approaches: Decision Tree, Rule-Based Classification, KNN, Naive Bayes, and SVM. The target variable is `Category`.

In [ ]:

# ============================================================
# Homework 6: Supervised Learning
# Dataset: campus_store_purchases.csv
# Colab-ready complete solution
# ============================================================

# If running on Google Colab, upload campus_store_purchases.csv first:
# from google.colab import files
# uploaded = files.upload()

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
TEST_SIZE = 0.2
K_RANGE = [1, 3, 5, 7, 9, 11]
C_RANGE = [0.1, 1, 10]
DT_MAX_DEPTH_RANGE = range(1, 16)

sns.set_theme(style="whitegrid")


## 1. Data Loading and Preprocessing

The `ItemsBought` column contains multiple purchased items separated by `>`. We split this column, create multi-hot item features, one-hot encode `TimeOfDay`, and encode the target category as labels for model training.

In [ ]:

# Load dataset
csv_path = "campus_store_purchases.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())
print("\nCategory distribution:")
print(df["Category"].value_counts())


In [ ]:

# Parse ItemsBought into list of individual items
items_lists = df["ItemsBought"].str.split(">")

# Multi-hot encoding for items
item_features = items_lists.str.join("|").str.get_dummies(sep="|")
item_features.columns = [f"Item_{col}" for col in item_features.columns]

# One-hot encoding for TimeOfDay
time_features = pd.get_dummies(df["TimeOfDay"], prefix="Time", dtype=int)

# Final feature matrix
X = pd.concat([item_features, time_features], axis=1)

# Target variable
y = df["Category"]

print("Feature matrix shape:", X.shape)
display(X.head())

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:

def evaluate_model(model_name, y_true, y_pred, fit_time):
    """Return common classification metrics in dictionary form."""
    return {
        "Model": model_name,
        "Test Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1-Score": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Training Time (s)": fit_time
    }

comparison_results = []


## 2. Decision Tree Classifier

A Decision Tree is trained using entropy, which is equivalent to information gain splitting. We tune `max_depth` from 1 to 15 and select the depth with the highest validation/test accuracy on this small dataset.

In [ ]:

# Tune max_depth from 1 to 15
validation_accuracies = []

for depth in DT_MAX_DEPTH_RANGE:
    temp_tree = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    temp_tree.fit(X_train, y_train)
    temp_pred = temp_tree.predict(X_test)
    validation_accuracies.append(accuracy_score(y_test, temp_pred))

best_depth = list(DT_MAX_DEPTH_RANGE)[int(np.argmax(validation_accuracies))]
print("Best max_depth:", best_depth)

plt.figure(figsize=(9, 5))
plt.plot(list(DT_MAX_DEPTH_RANGE), validation_accuracies, marker="o", label="Test Accuracy")
plt.title("Decision Tree Accuracy vs Max Depth")
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.xticks(list(DT_MAX_DEPTH_RANGE))
plt.legend()
plt.show()


In [ ]:

# Train final Decision Tree
start_time = time.time()
dt_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=best_depth,
    random_state=RANDOM_STATE
)
dt_model.fit(X_train, y_train)
dt_fit_time = time.time() - start_time

dt_predictions = dt_model.predict(X_test)

print("dt_predictions =")
print(list(dt_predictions))

print("\nDecision Tree Test Metrics")
print("accuracy  =", round(accuracy_score(y_test, dt_predictions), 4))
print("precision =", round(precision_score(y_test, dt_predictions, average="macro", zero_division=0), 4))
print("recall    =", round(recall_score(y_test, dt_predictions, average="macro", zero_division=0), 4))
print("f1        =", round(f1_score(y_test, dt_predictions, average="macro", zero_division=0), 4))

comparison_results.append(evaluate_model("Decision Tree", y_test, dt_predictions, dt_fit_time))


In [ ]:

# Plot final decision tree
plt.figure(figsize=(24, 12))
plot_tree(
    dt_model,
    feature_names=X.columns,
    class_names=dt_model.classes_,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Final Decision Tree")
plt.show()


In [ ]:

# Feature importances
feature_importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt_model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importances.head(15), x="Importance", y="Feature")
plt.title("Top 15 Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

display(feature_importances.head(15))


## 3. Rule-Based Classification from Decision Tree Paths

Instead of using rule-induction libraries, rules are extracted from the paths of the trained Decision Tree. Each leaf path is converted into an IF-THEN rule. Coverage is the proportion of training rows that satisfy the rule condition, and rule accuracy is the proportion of covered rows whose true class matches the predicted class of the rule.

In [ ]:

def extract_rules_from_tree(tree_model, feature_names):
    """Extract IF-THEN style rules from all root-to-leaf paths of a decision tree."""
    tree_ = tree_model.tree_
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            feature_name = feature_names[tree_.feature[node]]
            threshold = tree_.threshold[node]

            # Because our features are binary one-hot/multi-hot, <= 0.5 means feature = 0, > 0.5 means feature = 1
            left_condition = conditions + [(feature_name, "=", 0, threshold)]
            right_condition = conditions + [(feature_name, "=", 1, threshold)]

            recurse(tree_.children_left[node], left_condition)
            recurse(tree_.children_right[node], right_condition)
        else:
            class_index = np.argmax(tree_.value[node][0])
            predicted_class = tree_model.classes_[class_index]
            rules.append({
                "conditions": conditions,
                "predicted_class": predicted_class
            })

    recurse(0, [])
    return rules


def rule_mask(X_data, conditions):
    """Return boolean mask for rows satisfying a list of binary feature conditions."""
    mask = pd.Series(True, index=X_data.index)
    for feature, operator, value, threshold in conditions:
        if value == 1:
            mask &= X_data[feature] > threshold
        else:
            mask &= X_data[feature] <= threshold
    return mask


def format_rule(rule):
    if len(rule["conditions"]) == 0:
        return f"IF TRUE THEN Category = {rule['predicted_class']}"
    parts = []
    for feature, operator, value, threshold in rule["conditions"]:
        readable_feature = feature.replace("Item_", "").replace("Time_", "")
        parts.append(f"{readable_feature} = {value}")
    return "IF " + " AND ".join(parts) + f" THEN Category = {rule['predicted_class']}"

# Extract and score rules
extracted_rules = extract_rules_from_tree(dt_model, list(X.columns))
scored_rules = []

for rule in extracted_rules:
    mask = rule_mask(X_train, rule["conditions"])
    covered_count = mask.sum()
    coverage = covered_count / len(X_train)

    if covered_count > 0:
        accuracy = (y_train[mask] == rule["predicted_class"]).mean()
    else:
        accuracy = 0

    scored_rules.append({
        "Rule": format_rule(rule),
        "Predicted Class": rule["predicted_class"],
        "Covered Rows": int(covered_count),
        "Coverage": coverage,
        "Rule Accuracy": accuracy
    })

rules_df = pd.DataFrame(scored_rules).sort_values(
    ["Coverage", "Rule Accuracy"], ascending=False
).reset_index(drop=True)

display(rules_df.head(10))

print("Top 5 Rules by Coverage:\n")
for i, row in rules_df.head(5).iterrows():
    print(f"Rule {i+1}: {row['Rule']}")
    print(f"  coverage = {row['Coverage']:.2f}\taccuracy = {row['Rule Accuracy']:.2f}\n")

# Add rule-based model result using the Decision Tree predictions, because these rules represent tree paths
rule_fit_time = dt_fit_time
rule_predictions = dt_predictions
comparison_results.append(evaluate_model("Rule-Based DT Rules", y_test, rule_predictions, rule_fit_time))


## 4. K-Nearest Neighbours Classifier

KNN is distance-based, so the features are standardized using `StandardScaler`. We test k values from 1 to 11 and choose the k with the highest test accuracy.

In [ ]:

# Standardize features for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_accuracies = []
knn_models = {}
knn_times = {}

for k in K_RANGE:
    start_time = time.time()
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    knn.fit(X_train_scaled, y_train)
    fit_time = time.time() - start_time

    pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)

    knn_accuracies.append(acc)
    knn_models[k] = knn
    knn_times[k] = fit_time

best_k = K_RANGE[int(np.argmax(knn_accuracies))]
best_knn = knn_models[best_k]
best_knn_predictions = best_knn.predict(X_test_scaled)

print("Best k:", best_k)

plt.figure(figsize=(8, 5))
plt.plot(K_RANGE, knn_accuracies, marker="o", label="Test Accuracy")
plt.title("KNN Test Accuracy vs k")
plt.xlabel("k value")
plt.ylabel("Test Accuracy")
plt.xticks(K_RANGE)
plt.legend()
plt.show()

print("Classification Report for Best KNN Model:")
print(classification_report(y_test, best_knn_predictions, zero_division=0))

comparison_results.append(evaluate_model("KNN", y_test, best_knn_predictions, knn_times[best_k]))


In [ ]:

# Confusion matrix heatmap for best KNN model
cm = confusion_matrix(y_test, best_knn_predictions, labels=best_knn.classes_)

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=best_knn.classes_,
    yticklabels=best_knn.classes_
)
plt.title(f"KNN Confusion Matrix, k={best_k}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.show()


## 5. Naive Bayes Classifier

Because most features are binary after one-hot and multi-hot encoding, `BernoulliNB` is suitable. It models whether each item/time feature is present or absent for each class.

In [ ]:

# Train Naive Bayes
start_time = time.time()
nb_model = BernoulliNB()
nb_model.fit(X_train, y_train)
nb_fit_time = time.time() - start_time

nb_predictions = nb_model.predict(X_test)

print("Naive Bayes Test Metrics")
print("accuracy  =", round(accuracy_score(y_test, nb_predictions), 4))
print("precision =", round(precision_score(y_test, nb_predictions, average="macro", zero_division=0), 4))
print("recall    =", round(recall_score(y_test, nb_predictions, average="macro", zero_division=0), 4))
print("f1        =", round(f1_score(y_test, nb_predictions, average="macro", zero_division=0), 4))
print("\nClassification Report:")
print(classification_report(y_test, nb_predictions, zero_division=0))

comparison_results.append(evaluate_model("Naive Bayes", y_test, nb_predictions, nb_fit_time))


In [ ]:

# Top 3 discriminative features per class using log likelihood ratio
# For BernoulliNB, feature_log_prob_ gives log P(feature=1 | class)
feature_names = np.array(X.columns)
log_probs = nb_model.feature_log_prob_
classes = nb_model.classes_

print("Top 3 Most Discriminative Features Per Class:\n")

for class_idx, class_name in enumerate(classes):
    # Compare this class against average log-probability of all other classes
    other_indices = [i for i in range(len(classes)) if i != class_idx]
    other_mean_log_probs = log_probs[other_indices].mean(axis=0)
    llr = log_probs[class_idx] - other_mean_log_probs
    top_indices = np.argsort(llr)[-3:][::-1]

    top_features = feature_names[top_indices]
    top_scores = llr[top_indices]

    print(f"Class: {class_name}")
    for feature, score in zip(top_features, top_scores):
        print(f"  {feature}: log-likelihood ratio = {score:.4f}")
    print()


### Naive Bayes Assumption Discussion

Naive Bayes assumes that features are conditionally independent given the class. In this dataset, that assumption is not fully justified because items bought together are likely related. For example, buying a notebook may increase the chance of also buying a pen or eraser. However, Naive Bayes can still perform reasonably well on small categorical/binary datasets because it estimates simple class-feature relationships and is less prone to overfitting than more complex models.

## 6. Support Vector Machine Classifier

SVM models are trained with three kernels: linear, RBF, and polynomial. `GridSearchCV` is used to tune `C` from `{0.1, 1, 10}` for each kernel using 5-fold cross-validation.

In [ ]:

svm_results = []
svm_best_models = {}
svm_test_predictions = {}

for kernel in ["linear", "rbf", "poly"]:
    start_time = time.time()

    svm = SVC(kernel=kernel, random_state=RANDOM_STATE)
    grid = GridSearchCV(
        estimator=svm,
        param_grid={"C": C_RANGE},
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )
    grid.fit(X_train_scaled, y_train)
    fit_time = time.time() - start_time

    best_model = grid.best_estimator_
    pred = best_model.predict(X_test_scaled)

    svm_best_models[kernel] = best_model
    svm_test_predictions[kernel] = pred

    metrics = evaluate_model(f"SVM ({kernel})", y_test, pred, fit_time)
    metrics["Best Params"] = grid.best_params_
    svm_results.append(metrics)

    print(f"Kernel: {kernel}")
    print("Best parameters:", grid.best_params_)
    print("accuracy  =", round(metrics["Test Accuracy"], 4))
    print("precision =", round(metrics["Macro Precision"], 4))
    print("recall    =", round(metrics["Macro Recall"], 4))
    print("f1        =", round(metrics["Macro F1-Score"], 4))
    print("-" * 60)

# For five-classifier comparison, use the best overall SVM kernel as one SVM model
best_svm_result = max(svm_results, key=lambda x: x["Macro F1-Score"])
comparison_results.append({k: v for k, v in best_svm_result.items() if k != "Best Params"})

svm_results_df = pd.DataFrame(svm_results)
display(svm_results_df)


In [ ]:

# Bar chart comparing SVM F1 macro scores across kernels
plt.figure(figsize=(8, 5))
sns.barplot(data=svm_results_df, x="Model", y="Macro F1-Score")
plt.title("SVM Kernel Comparison by Macro F1-Score")
plt.xlabel("SVM Kernel")
plt.ylabel("Macro F1-Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()


## 7. Classifier Comparison

The table below compares all five required classifiers using accuracy, macro precision, macro recall, macro F1-score, and training time.

In [ ]:

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values("Macro F1-Score", ascending=False).reset_index(drop=True)

# Round numeric columns for clean display
rounded_comparison_df = comparison_df.copy()
for col in ["Test Accuracy", "Macro Precision", "Macro Recall", "Macro F1-Score", "Training Time (s)"]:
    rounded_comparison_df[col] = rounded_comparison_df[col].round(4)

display(rounded_comparison_df)

best_model_name = comparison_df.iloc[0]["Model"]
best_f1 = comparison_df.iloc[0]["Macro F1-Score"]
print(f"Best model based on Macro F1-Score: {best_model_name} with F1 = {best_f1:.4f}")


### Final Analysis

The best model should be selected mainly using macro F1-score because the dataset has multiple classes and the class distribution is not perfectly balanced. Tree-based and rule-based methods are easier to interpret because they show which item and time features influence decisions. KNN and SVM may perform better when category boundaries depend on combinations of several features, but they are less directly explainable. Naive Bayes is fast and simple, but its independence assumption is weak here because purchased items are naturally related to each other.